In [31]:
import os
import sys
import random
import subprocess
import json
from typing import List, Dict, Tuple, Optional
import torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    """Пытается импортировать пакет и при необходимости установить его через pip."""
    target = import_name or package_name
    try:
        __import__(target)
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


# Для retrieval-контура попробуем установить основные зависимости.
# Даже если sentence-transformers не поднимется, ноутбук сможет работать через fallback.
ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")


try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False
    print("FAISS недоступен, будет использован fallback на sklearn NearestNeighbors.")
    print("Причина:", repr(e))


print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS available:", FAISS_AVAILABLE)

os.makedirs("./artifacts", exist_ok=True)
pd.set_option('display.max_colwidth', 1000)
pd.set_option('display.width', 1500)
pd.set_option('display.max_columns', None)

NumPy: 2.0.2
Pandas: 2.2.2
FAISS available: True


In [32]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

try:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)

Устройство для работы: cpu


In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
documents = json.load(open("science.json", encoding="utf-8"))["documents"]

In [35]:
df_documents = pd.DataFrame(documents)
df_documents.head()

,doc_id,title,author,year,text
0,science_01,Теория относительности,Альберт Эйнштейн,1915,"Фундаментальная физическая теория пространства-времени и гравитации, пришедшая на смену ньютоновской механике. Специальная теория относительности, опубликованная в 1905 году, постулирует, что скорость света в вакууме постоянна и не зависит от движения источника или наблюдателя. Из этого следуют удивительные выводы: время замедляется при движении, длина сокращается, а масса растёт со скоростью. Знаменитая формула E=mc² показывает эквивалентность массы и энергии. Общая теория относительности, завершённая в 1915 году, описывает гравитацию как искривление четырёхмерного пространства-времени массивными телами. Именно это искривление заставляет планеты вращаться по орбитам, а свет отклоняться вблизи Солнца. Теория предсказала существование чёрных дыр — областей, откуда ничто не может вырваться, даже свет, а также гравитационных волн, обнаруженных только в 2015 году. Без учёта релятивистских эффектов не работали бы системы GPS, расходясь на несколько километров в день."
1,science_02,Естественный отбор,Чарльз Дарвин,1859,"Основной механизм эволюции живых организмов, объясняющий, как из простых форм постепенно возникают сложные виды. Идея заключается в трёх простых фактах: особи одного вида различаются по наследственным признакам; потомства рождается больше, чем может выжить; выживают и размножаются те, чьи признаки лучше приспособлены к окружающей среде. За многие поколения полезные мутации накапливаются, и вид постепенно меняется. Дарвин совершил кругосветное путешествие на корабле «Бигль», где особенно его поразили вьюрки на Галапагосских островах: у разных видов клювы были приспособлены к разной пище — семенам, насекомым или кактусам. Теория вызвала бурные споры с религиозными кругами, так как противоречила буквальному прочтению Библии о сотворении мира за шесть дней. Сегодня теория эволюции подтверждена данными генетики, палеонтологии и наблюдениями за развитием антибиотикорезистентности у бактерий. Например, за 70 лет бактерии выработали устойчивость почти ко всем антибиотикам, что является наг..."
2,science_03,Строение ДНК,Джеймс Уотсон и Фрэнсис Крик,1953,"Расшифровка двойной спирали дезоксирибонуклеиновой кислоты, ставшая краеугольным камнем современной молекулярной биологии. Молекула ДНК состоит из двух длинных цепочек, закрученных друг вокруг друга наподобие винтовой лестницы. Каждая цепочка строится из четырёх типов нуклеотидов: аденина (А), тимина (Т), гуанина (Г) и цитозина (Ц). Секрет наследственности кроется в принципе комплементарности: А всегда соединяется с Т, а Г — с Ц, подобно ключу к замку. Это означает, что каждая из двух цепочек хранит точную копию информации и может служить матрицей для восстановления второй половинки. Уотсон и Крик использовали рентгеновские снимки Розалинд Франклин, на которых была видна характерная спиральная форма. Открытие объяснило, как происходит репликация ДНК перед делением клетки и как возникают мутации при ошибках копирования. Сегодня расшифровка генома человека занимает всего несколько дней, а технологии CRISPR позволяют редактировать гены, исправляя наследственные болезни. Без понимания ..."
3,science_04,Законы Ньютона,Исаак Ньютон,1687,"Три фундаментальных закона, лежащих в основе классической механики и описания движения тел. Первый закон (закон инерции) гласит: тело сохраняет состояние покоя или равномерного прямолинейного движения, пока на него не подействуют другие силы. Второй закон — самый известный: сила равна массе, умноженной на ускорение (F=ma). Именно он объясняет, почему тяжёлый грузовик разгоняется медленнее легкового автомобиля при той же силе двигателя. Третий закон: действие равно противодействию. Когда вы прыгаете вверх, вы отталкиваете Землю вниз, но из-за огромной массы планеты её смещение незаметно. Ньютон опубликовал эти законы в своём главном труде «Математические начала натуральной философии», где также сформулировал закон всемирного тяготения. Легенда о яблоке, упавшем

In [36]:
df_documents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   doc_id  15 non-null     object
 1   title   15 non-null     object
 2   author  15 non-null     object
 3   year    15 non-null     object
 4   text    15 non-null     object
dtypes: object(5)
memory usage: 732.0+ bytes


In [37]:
class SentenceTransformersBackend:
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")


embedder = SentenceTransformersBackend(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=DEVICE
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [38]:
def chunk_text(text: str, chunk_size: int = 30, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks

In [39]:
def build_chunks_dataframe(
    docs: List[Dict[str, str]],
    chunk_size: int = 10,
    overlap: int = 5,
) -> pd.DataFrame:
    rows = []

    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk in enumerate(chunks):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": chunk_id,
                    "chunk_text": chunk,
                    "n_words": len(chunk.split()),
                }
            )

    return pd.DataFrame(rows)


chunks_df = build_chunks_dataframe(documents, chunk_size=22, overlap=5)

print("Количество чанков:", len(chunks_df))
display(chunks_df.head(10))

Количество чанков: 136


,doc_id,title,chunk_id,chunk_text,n_words
0,science_01,Теория относительности,0,"Фундаментальная физическая теория пространства-времени и гравитации, пришедшая на смену ньютоновской механике. Специальная теория относительности, опубликованная в 1905 году, постулирует, что скорость света",22
1,science_01,Теория относительности,1,"году, постулирует, что скорость света в вакууме постоянна и не зависит от движения источника или наблюдателя. Из этого следуют удивительные выводы: время",22
2,science_01,Теория относительности,2,"этого следуют удивительные выводы: время замедляется при движении, длина сокращается, а масса растёт со скоростью. Знаменитая формула E=mc² показывает эквивалентность массы и",22
3,science_01,Теория относительности,3,"E=mc² показывает эквивалентность массы и энергии. Общая теория относительности, завершённая в 1915 году, описывает гравитацию как искривление четырёхмерного пространства-времени массивными телами. Именно",22
4,science_01,Теория относительности,4,"четырёхмерного пространства-времени массивными телами. Именно это искривление заставляет планеты вращаться по орбитам, а свет отклоняться вблизи Солнца. Теория предсказала существование чёрных дыр",22
5,science_01,Теория относительности,5,"Теория предсказала существование чёрных дыр — областей, откуда ничто не может вырваться, даже свет, а также гравитационных волн, обнаруженных только в 2015",22
6,science_01,Теория относительности,6,"волн, обнаруженных только в 2015 году. Без учёта релятивистских эффектов не работали бы системы GPS, расходясь на несколько километров в день.",21
7,science_02,Естественный отбор,0,"Основной механизм эволюции живых организмов, объясняющий, как из простых форм постепенно возникают сложные виды. Идея заключается в трёх простых фактах: особи одного",22
8,science_02,Естественный отбор,1,"трёх простых фактах: особи одного вида различаются по наследственным признакам; потомства рождается больше, чем может выжить; выживают и размножаются те, чьи признаки",22
9,science_02,Естественный отбор,2,"и размножаются те, чьи признаки лучше приспособлены к окружающей среде. За многие поколения полезные мутации накапливаются, и вид постепенно меняется. Дарвин совершил",22


In [40]:
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunk_texts)

chunk_embeddings

array([[-0.09211369,  0.03486871,  0.0069332 , ..., -0.02218975,
        -0.04279288,  0.01953562],
       [-0.10438668,  0.0803108 ,  0.04103775, ...,  0.0139048 ,
        -0.02221327,  0.00417542],
       [-0.07447113,  0.04819288,  0.05412608, ..., -0.04666879,
         0.02720088,  0.00609988],
       ...,
       [-0.09511378,  0.09556542, -0.06807909, ..., -0.07315653,
         0.120075  , -0.00075854],
       [-0.11669356,  0.06021896, -0.05275242, ...,  0.00534855,
         0.11718971, -0.02160222],
       [-0.09392223,  0.06163593, -0.00118578, ...,  0.04550907,
         0.08393916, -0.06391859]], dtype=float32)

In [41]:
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self.backend_name = None
        self._faiss_index = None
        self._nn_index = None
        self._faiss_index = faiss.IndexFlatIP(dim)
        self.backend_name = "FAISS IndexFlatIP"

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")
        self._faiss_index.add(vectors)

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")
        scores, indices = self._faiss_index.search(query_vectors, top_k)
        return scores, indices


search_index = VectorSearchIndex(dim=chunk_embeddings.shape[1])
search_index.add(chunk_embeddings)

In [42]:
def search_similar_chunks(query: str, top_k: int = 5) -> pd.DataFrame:
    query_vectors = embedder.encode_queries([query])
    scores, indices = search_index.search(query_vectors, top_k=top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": int(chunk_row["chunk_id"]),
                "score": round(float(score), 4),
                "chunk_text": chunk_row["chunk_text"],
            }
        )

    return pd.DataFrame(rows)

In [43]:
questions = {
    # science_01 - Теория относительности
    "В каком году опубликована специальная теория относительности Эйнштейна, и какое предсказание подтвердили только в 2015 году?": "science_01",

    # science_02 - Естественный отбор
    "Как назывался корабль Дарвина, и какие птицы на Галапагоссах поразили его разнообразием клювов?": "science_02",

    # science_03 - Строение ДНК
    "Кто сделал рентгеновские снимки ДНК для Уотсона и Крика, и как называется технология редактирования генов?": "science_03",

    # science_04 - Законы Ньютона
    "В каком году вышли «Математические начала», и какая формула выражает второй закон Ньютона?": "science_04",

    # science_05 - Периодический закон
    "Какие три элемента предсказал Менделеев, и под какими временными названиями (с «эка-») они фигурировали?": "science_05",

    # science_06 - Теория большого взрыва
    "Сколько миллиардов лет назад произошёл Большой взрыв, и через сколько тысяч лет возникло реликтовое излучение?": "science_06",

    # science_07 - Квантовая механика
    "Как называется мысленный эксперимент с котом, и какой принцип гласит о невозможности точно измерить положение и импульс частицы?": "science_07",

    # science_08 - Континентальный дрейф
    "Как назывался суперконтинент 300 млн лет назад, и с какой скоростью (см/год) движутся тектонические плиты?": "science_08",

    # science_09 - Вакцинация
    "Как звали мальчика — первого привитого, и какую болезнь полностью искоренили к 1980 году?": "science_09",

    # science_10 - Гелиоцентрическая система
    "В каком году вышел труд Коперника, и кто из учёных подтвердил его теорию, но предстал перед инквизицией?": "science_10",

    # science_11 - Пенициллин
    "Какая плесень дала первый антибиотик, и в каком году Флеминг получил Нобелевскую премию?": "science_11",

    # science_12 - Электромагнетизм
    "Из скольких уравнений состоит система Максвелла, и кто экспериментально доказал существование радиоволн?": "science_12",

    # science_13 - Гомеостаз
    "Кто ввёл термин «гомеостаз», и как Клод Бернар назвал внутреннюю среду организма?": "science_13",

    # science_14 - Радиоактивность
    "Какой элемент Мария Кюри назвала в честь Польши, и от какой болезни она умерла?": "science_14",

    # science_15 - Кислородная теория горения
    "В каком году Лавуазье опубликовал теорию горения, и в каком году он был казнён?": "science_15",
}
for current_query in questions:
    display(Markdown(f"### Запрос: `{current_query}`"))
    display(search_similar_chunks(current_query, top_k=3))

### Запрос: `В каком году опубликована специальная теория относительности Эйнштейна, и какое предсказание подтвердили только в 2015 году?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,science_01,Теория относительности,0,0.6151,"Фундаментальная физическая теория пространства-времени и гравитации, пришедшая на смену ньютоновской механике. Специальная теория относительности, опубликованная в 1905 году, постулирует, что скорость света"
1,2,science_04,Законы Ньютона,7,0.6036,"до движения планет. Однако на очень больших скоростях (близких к скорости света) они уступают место теории относительности Эйнштейна, а на атомном уровне"
2,3,science_01,Теория относительности,3,0.5816,"E=mc² показывает эквивалентность массы и энергии. Общая теория относительности, завершённая в 1915 году, описывает гравитацию как искривление четырёхмерного пространства-времени массивными телами. Именно"


### Запрос: `Как назывался корабль Дарвина, и какие птицы на Галапагоссах поразили его разнообразием клювов?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,science_02,Естественный отбор,3,0.8306,"вид постепенно меняется. Дарвин совершил кругосветное путешествие на корабле «Бигль», где особенно его поразили вьюрки на Галапагосских островах: у разных видов клювы"
1,2,science_02,Естественный отбор,4,0.5957,"островах: у разных видов клювы были приспособлены к разной пище — семенам, насекомым или кактусам. Теория вызвала бурные споры с религиозными кругами,"
2,3,science_08,Континентальный дрейф,2,0.4619,пазла. Он собрал доказательства: окаменелости одного и того же древнего пресмыкающегося мезозавра найдены только в Бразилии и Южной Африке — переплыть солёный


### Запрос: `Кто сделал рентгеновские снимки ДНК для Уотсона и Крика, и как называется технология редактирования генов?`

...

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,science_13,Гомеостаз,2,0.7531,он назвал «milieu intérieur» (внутренняя среда). Постоянство этой среды — залог здоровой жизни. Термин «гомеостаз» предложил американский физиолог Уолтер Кэннон в 1926
1,2,science_13,Гомеостаз,0,0.5570,"Концепция, описывающая способность живых организмов поддерживать постоянство внутренней среды, несмотря на изменения внешних условий. Французский физиолог Клод Бернар первым заметил, что все"
2,3,science_13,Гомеостаз,1,0.5453,"Бернар первым заметил, что все клетки тела живут в жидкости — крови, лимфе и межклеточной жидкости, которую он назвал «milieu intérieur» (внутренняя"


### Запрос: `Какой элемент Мария Кюри назвала в честь Польши, и от какой болезни она умерла?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,science_14,Радиоактивность,5,0.5341,"светом. Кюри работали без всякой защиты, не зная об опасности радиации. Мария умерла от апластической анемии, вызванной облучением, а её тетради до"
1,2,science_14,Радиоактивность,2,0.4553,чёрную бумагу. Однако именно Мария Кюри и её муж Пьер ввели сам термин «радиоактивность» и совершили подвиг науки: переработав тонны урановой руды
2,3,science_09,Вакцинация,2,0.4165,"заболевали смертельно опасной натуральной оспой, которая косила целые города. Дженнер взял содержимое пустулы с руки доярки Сары Нелмс и втёр его в"


### Запрос: `В каком году Лавуазье опубликовал теорию горения, и в каком году он был казнён?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,science_15,Кислородная теория горения,0,0.6227,"Научное объяснение процесса горения и дыхания, опровергнувшее господствовавшую более ста лет теорию флогистона — воображаемой субстанции, якобы выделяющейся при горении. Лавуазье провёл"
1,2,science_10,Гелиоцентрическая система,5,0.5491,"Коперник опубликовал свой труд «О вращениях небесных сфер» только в год смерти, получив первый экземпляр на смертном одре. Позже Галилео Галилей, наблюдая"
2,3,science_15,Кислородная теория горения,1,0.4873,"выделяющейся при горении. Лавуазье провёл серию блестящих количественных экспериментов: он нагревал металлы (ртуть, олово, свинец) в герметичных сосудах и измерял общий вес"


In [44]:
results_comparison = []

for top_k in [2, 7]:
    print(f"\n--- Тест: top_k={top_k} ---")

    hits = 0
    mrr_sum = 0

    for query, expected in questions.items():
        answer_df = search_similar_chunks(query, top_k=top_k)
        retrieved_docs = answer_df["doc_id"].tolist()

        # Hit@k и Recall@k (в данном случае они равны, т.к. один релевантный документ)
        if expected in retrieved_docs:
            hits += 1

        # MRR@k
        rank = None
        for i, doc_id in enumerate(retrieved_docs):
            if doc_id == expected:
                rank = i + 1
                break
        if rank:
            mrr_sum += 1 / rank

    hit_rate = hits / len(questions)
    recall = hits / len(questions)  # recall = hit_rate, т.к. 1 релевантный документ
    mrr = mrr_sum / len(questions)

    print(f"  Hit@{top_k}:    {hits}/{len(questions)} ({hit_rate*100:.1f}%)")
    print(f"  Recall@{top_k}: {recall:.3f}")
    print(f"  MRR@{top_k}:    {mrr:.3f}")

    results_comparison.append({
        "top_k": top_k,
        "hit": hits,
        "hit_rate": round(hit_rate, 3),
        "recall": round(recall, 3),
        "mrr": round(mrr, 3)
    })

print("\n" + "=" * 60)
print("РЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТА")
print("=" * 60)

comparison_df = pd.DataFrame(results_comparison)
comparison_df.to_csv("./artifacts/top_k_comparison.csv", index=False)
display(comparison_df)


--- Тест: top_k=2 ---
  Hit@2:    15/15 (100.0%)
  Recall@2: 1.000
  MRR@2:    0.967

--- Тест: top_k=7 ---
  Hit@7:    15/15 (100.0%)
  Recall@7: 1.000
  MRR@7:    0.967

РЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТА


,top_k,hit,hit_rate,recall,mrr
0,2,15,1.0,1.0,0.967
1,7,15,1.0,1.0,0.967


In [45]:
results_top_2 = []

for query, expected in questions.items():
    answer_df = search_similar_chunks(query, top_k=2)
    retrieved_docs = answer_df["doc_id"].tolist()

    hit = int(expected in retrieved_docs)

    rank = None
    for i, doc_id in enumerate(retrieved_docs):
        if doc_id == expected:
            rank = i + 1
            break

    recall = hit

    mrr = 1 / rank if rank else 0

    results_top_2.append({
        "query": query,
        "expected_source": expected,
        "retrieved_sources": retrieved_docs,
        "hit_at_k": hit,
        "recall_at_k": recall,
        "rank_of_first_relevant": rank,
        "mrr_at_k": round(mrr, 3)
    })

retrieval_eval = pd.DataFrame(results_top_2)
retrieval_eval.to_csv("./artifacts/retrieval_eval.csv")
hit_at_2 = retrieval_eval["hit_at_k"].sum()
print(f"hit@2: {hit_at_2}/15 = {hit_at_2/15*100:.1f}%")
mrr_sum = retrieval_eval["mrr_at_k"].sum()
print(f"MRR@2: {mrr_sum/15*100:.1f}%")
print(f"recall@2: {hit_at_2/15*100:.1f}%")
ranks = retrieval_eval["rank_of_first_relevant"].dropna()
print(f"\nРаспределение рангов:")
print(retrieval_eval["rank_of_first_relevant"].value_counts().sort_index())
retrieval_eval

hit@2: 15/15 = 100.0%
MRR@2: 96.7%
recall@2: 100.0%

Распределение рангов:
rank_of_first_relevant
1    14
2     1
Name: count, dtype: int64


,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,rank_of_first_relevant,mrr_at_k
0,"В каком году опубликована специальная теория относительности Эйнштейна, и какое предсказание подтвердили только в 2015 году?",science_01,"[science_01, science_04]",1,1,1,1.0
1,"Как назывался корабль Дарвина, и какие птицы на Галапагоссах поразили его разнообразием клювов?",science_02,"[science_02, science_02]",1,1,1,1.0
2,"Кто сделал рентгеновские снимки ДНК для Уотсона и Крика, и как называется технология редактирования генов?",science_03,"[science_03, science_03]",1,1,1,1.0
3,"В каком году вышли «Математические начала», и какая формула выражает второй закон Ньютона?",science_04,"[science_10, science_04]",1,1,2,0.5
4,"Какие три элемента предсказал Менделеев, и под какими временными названиями (с «эка-») они фигурировали?",science_05,"[science_05, science_05]",1,1,1,1.0
5,"Сколько миллиардов лет назад произошёл Большой взрыв, и через сколько тысяч лет возникло реликтовое излучение?",science_06,"[science_06, science_06]",1,1,1,1.0
6,"Как называется мысленный эксперимент с котом, и какой принцип гласит о невозможности точно измерить положение и импульс частицы?",science_07,"[science_07, science_07]",1,1,1,1.0
7,"Как назывался суперконтинент 300 млн лет назад, и с какой скоростью (см/год) движутся тектонические плиты?",science_08,"[science_08, science_08]",1,1,1,1.0
8,"Как звали мальчика — первого привитого, и какую болезнь полностью искоренили к 1980 году?",science_09,"[science_09, science_09]",1,1,1,1.0
9,"В каком году вышел труд Коперника, и кто из учёных подтвердил его теорию, но предстал перед инквизицией?",science_10,"[science_10, science_10]",1,1,1,1.0


In [46]:
results_top_7 = []

for query, expected in questions.items():
    answer_df = search_similar_chunks(query, top_k=7)
    retrieved_docs = answer_df["doc_id"].tolist()

    hit = int(expected in retrieved_docs)

    rank = None
    for i, doc_id in enumerate(retrieved_docs):
        if doc_id == expected:
            rank = i + 1
            break

    recall = hit

    mrr = 1 / rank if rank else 0

    results_top_7.append({
        "query": query,
        "expected_source": expected,
        "retrieved_sources": retrieved_docs,
        "hit_at_k": hit,
        "recall_at_k": recall,
        "rank_of_first_relevant": rank,
        "mrr_at_k": round(mrr, 3)
    })

retrieval_eval = pd.DataFrame(results_top_7)

hit_at_7 = retrieval_eval["hit_at_k"].sum()
print(f"hit@7: {hit_at_7}/15 = {hit_at_7/15*100:.1f}%")
mrr_sum = retrieval_eval["mrr_at_k"].sum()
print(f"MRR@7: {mrr_sum/15*100:.1f}%")
print(f"recall@7: {hit_at_7/15*100:.1f}%")
ranks = retrieval_eval["rank_of_first_relevant"].dropna()
print(f"\nРаспределение рангов:")
print(retrieval_eval["rank_of_first_relevant"].value_counts().sort_index())

retrieval_eval

hit@7: 15/15 = 100.0%
MRR@7: 96.7%
recall@7: 100.0%

Распределение рангов:
rank_of_first_relevant
1    14
2     1
Name: count, dtype: int64


,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,rank_of_first_relevant,mrr_at_k
0,"В каком году опубликована специальная теория относительности Эйнштейна, и какое предсказание подтвердили только в 2015 году?",science_01,"[science_01, science_04, science_01, science_01, science_04, science_06, science_10]",1,1,1,1.0
1,"Как назывался корабль Дарвина, и какие птицы на Галапагоссах поразили его разнообразием клювов?",science_02,"[science_02, science_02, science_08, science_06, science_08, science_02, science_06]",1,1,1,1.0
2,"Кто сделал рентгеновские снимки ДНК для Уотсона и Крика, и как называется технология редактирования генов?",science_03,"[science_03, science_03, science_03, science_03, science_03, science_03, science_06]",1,1,1,1.0
3,"В каком году вышли «Математические начала», и какая формула выражает второй закон Ньютона?",science_04,"[science_10, science_04, science_04, science_04, science_10, science_01, science_01]",1,1,2,0.5
4,"Какие три элемента предсказал Менделеев, и под какими временными названиями (с «эка-») они фигурировали?",science_05,"[science_05, science_05, science_05, science_10, science_05, science_08, science_10]",1,1,1,1.0
5,"Сколько миллиардов лет назад произошёл Большой взрыв, и через сколько тысяч лет возникло реликтовое излучение?",science_06,"[science_06, science_06, science_08, science_06, science_06, science_14, science_14]",1,1,1,1.0
6,"Как называется мысленный эксперимент с котом, и какой принцип гласит о невозможности точно измерить положение и импульс частицы?",science_07,"[science_07, science_07, science_07, science_07, science_15, science_07, science_13]",1,1,1,1.0
7,"Как назывался суперконтинент 300 млн лет назад, и с какой скоростью (см/год) движутся тектонические плиты?",science_08,"[science_08, science_08, science_06, science_06, science_08, science_08, science_06]",1,1,1,1.0
8,"Как звали мальчика — первого привитого, и какую болезнь полностью искоренили к 1980 году?",science_09,"[science_09, science_09, science_11, science_09, science_09, science_09, science_09]",1,1,1,1.0
9,"В каком году вышел труд Коперника, и кто из учёных подтвердил его теорию, но предстал перед инквизицией?",science_10,"[science_10, science_10, science_10, science_10, science_06, science_15, science_05]",1,1,1,1.0


In [47]:
new_documents = [
    {
        "doc_id": "new_sci_01",
        "title": "Закон всемирного тяготения",
        "author": "Исаак Ньютон",
        "year": 1687,
        "text": "Фундаментальный закон физики, описывающий гравитационное взаимодействие между всеми телами, обладающими массой. Сила притяжения прямо пропорциональна произведению масс и обратно пропорциональна квадрату расстояния между ними. Легенда гласит, что Ньютон открыл этот закон, наблюдая за падением яблока в саду своего поместья. Закон объясняет движение планет вокруг Солнца, приливы и отливы на Земле, а также траектории полёта искусственных спутников."
    },
    {
        "doc_id": "new_sci_02",
        "title": "Эволюционная теория Ламарка",
        "author": "Жан-Батист Ламарк",
        "year": 1809,
        "text": "Первая целостная теория эволюции, согласно которой изменения в окружающей среде вызывают у организмов новые потребности, ведущие к изменению поведения. Ключевые принципы: упражнение или неупражнение органов и наследование приобретённых признаков. Классический пример — жирафы, которые якобы вытягивали шею, чтобы достать листья, и передавали эту особенность потомкам. Теория Ламарка позже была опровергнута генетикой."
    },
    {
        "doc_id": "new_sci_03",
        "title": "Открытие электрона",
        "author": "Джозеф Джон Томсон",
        "year": 1897,
        "text": "Экспериментальное доказательство существования первой элементарной частицы. Томсон изучал катодные лучи в стеклянной трубке и обнаружил, что они отклоняются электрическим и магнитным полями, что доказывало их отрицательный заряд. Он измерил отношение заряда к массе частицы и понял, что электрон примерно в 1800 раз легче атома водорода. За это открытие Томсон получил Нобелевскую премию в 1906 году."
    },
    {
        "doc_id": "new_sci_04",
        "title": "Правило правой руки",
        "author": "Джон Амброз Флеминг",
        "year": 1885,
        "text": "Мнемоническое правило в физике для определения направления электрического тока, магнитного поля или движения проводника. Согласно правилу, если расположить правую руку так, чтобы магнитные линии входили в ладонь, а отогнутый большой палец указывал направление движения проводника, то остальные четыре пальца покажут направление индукционного тока. Это правило широко используется при расчёте электродвигателей."
    }
]

update_questions = {
    "Что, согласно легенде, упало на голову Ньютону, натолкнув его на открытие закона всемирного тяготения?": "new_sci_01",

    "Какой именно пример (животное и его признак) приводит Ламарк в своей эволюционной теории как доказательство упражнения органов?": "new_sci_02",

    "Какую элементарную частицу открыл Джозеф Джон Томсон, и во сколько раз она легче атома водорода?": "new_sci_03",

    "Какую именно руку (правую или левую) использует правило Флеминга для определения направления индукционного тока?": "new_sci_04",
}

In [48]:
before_sources = []

for query, expected in update_questions.items():
    answer_df = search_similar_chunks(query, top_k=7)
    retrieved_docs = answer_df["doc_id"].tolist()
    before_sources.append(retrieved_docs)

In [49]:
original_documents = documents.copy()
documents_updated = documents + new_documents

In [50]:
old_chunks_len = len(chunks_df)

chunks_df_updated = build_chunks_dataframe(documents_updated, chunk_size=22, overlap=5)
print(f"Было чанков: {len(chunks_df)}")
print(f"Стало чанков: {len(chunks_df_updated)}")

Было чанков: 136
Стало чанков: 148


In [51]:
new_chunks = chunks_df_updated[chunks_df_updated["doc_id"].str.startswith("new_")]
new_chunks.head(5)

,doc_id,title,chunk_id,chunk_text,n_words
136,new_sci_01,Закон всемирного тяготения,0,"Фундаментальный закон физики, описывающий гравитационное взаимодействие между всеми телами, обладающими массой. Сила притяжения прямо пропорциональна произведению масс и обратно пропорциональна квадрату расстояния",22
137,new_sci_01,Закон всемирного тяготения,1,"и обратно пропорциональна квадрату расстояния между ними. Легенда гласит, что Ньютон открыл этот закон, наблюдая за падением яблока в саду своего поместья.",22
138,new_sci_01,Закон всемирного тяготения,2,"яблока в саду своего поместья. Закон объясняет движение планет вокруг Солнца, приливы и отливы на Земле, а также траектории полёта искусственных спутников.",22
139,new_sci_02,Эволюционная теория Ламарка,0,"Первая целостная теория эволюции, согласно которой изменения в окружающей среде вызывают у организмов новые потребности, ведущие к изменению поведения. Ключевые принципы: упражнение",22
140,new_sci_02,Эволюционная теория Ламарка,1,"изменению поведения. Ключевые принципы: упражнение или неупражнение органов и наследование приобретённых признаков. Классический пример — жирафы, которые якобы вытягивали шею, чтобы достать",22


In [52]:
chunk_texts_updated = chunks_df_updated["chunk_text"].tolist()
chunk_embeddings_updated = embedder.fit_documents(chunk_texts_updated)

In [53]:
print(f"Размерность старого индекса FAISS: {search_index._faiss_index.ntotal} векторов")
print(f"Количество чанков в старом DataFrame: {len(chunks_df)}")

search_index_updated = VectorSearchIndex(dim=chunk_embeddings_updated.shape[1])
search_index_updated.add(chunk_embeddings_updated)

print(f"Добавлено эмбеддингов в новый индекс: {chunk_embeddings_updated.shape[0]}")
print(f"Размерность нового индекса FAISS: {search_index_updated._faiss_index.ntotal} векторов")

chunks_df = chunks_df_updated
search_index = search_index_updated

print(f"  - Новых документов добавлено: {len(new_documents)}")
print(f"  - Новых чанков создано: {len(chunks_df_updated) - old_chunks_len}")
print(f"  - Общее количество чанков: {len(chunks_df)}")

Размерность старого индекса FAISS: 136 векторов
Количество чанков в старом DataFrame: 136
Добавлено эмбеддингов в новый индекс: 148
Размерность нового индекса FAISS: 148 векторов
  - Новых документов добавлено: 4
  - Новых чанков создано: 12
  - Общее количество чанков: 148


In [54]:
retrieval_after_update = []

for query, expected in update_questions.items():
    answer_df = search_similar_chunks(query, top_k=7)
    retrieved_docs = answer_df["doc_id"].tolist()

    hit = int(expected in retrieved_docs)

    rank = None
    for i, doc_id in enumerate(retrieved_docs):
        if doc_id == expected:
            rank = i + 1
            break

    recall = hit

    mrr = 1 / rank if rank else 0

    retrieval_after_update.append({
        "query": query,
        "expected_source": expected,
        "retrieved_sources": retrieved_docs,
        "hit@k": hit,
        "recall@k": recall,
        "rank_of_first_relevant": rank,
        "mrr@k": round(mrr, 3)
    })

retrieval_after_update = pd.DataFrame(retrieval_after_update)
retrieval_after_update

,query,expected_source,retrieved_sources,hit@k,recall@k,rank_of_first_relevant,mrr@k
0,"Что, согласно легенде, упало на голову Ньютону, натолкнув его на открытие закона всемирного тяготения?",new_sci_01,"[science_04, science_04, science_04, science_10, new_sci_01, science_10, science_08]",1,1,5,0.2
1,Какой именно пример (животное и его признак) приводит Ламарк в своей эволюционной теории как доказательство упражнения органов?,new_sci_02,"[new_sci_02, new_sci_02, science_02, science_13, science_04, new_sci_02, science_05]",1,1,1,1.0
2,"Какую элементарную частицу открыл Джозеф Джон Томсон, и во сколько раз она легче атома водорода?",new_sci_03,"[new_sci_03, science_15, new_sci_03, science_14, science_05, science_05, science_06]",1,1,1,1.0
3,Какую именно руку (правую или левую) использует правило Флеминга для определения направления индукционного тока?,new_sci_04,"[new_sci_04, new_sci_04, new_sci_04, science_04, science_04, science_12, science_04]",1,1,1,1.0


In [55]:
retrieval_before_after_update = pd.DataFrame()
retrieval_before_after_update["query"] = retrieval_after_update["query"]
retrieval_before_after_update["before_retrieved_sources"] = before_sources
retrieval_before_after_update["after_retrieved_sources"] = retrieval_after_update["retrieved_sources"]
retrieval_before_after_update["changed"] = retrieval_before_after_update["before_retrieved_sources"] != retrieval_before_after_update["after_retrieved_sources"]
retrieval_before_after_update.to_csv("./artifacts/retrieval_before_after_update.csv")
retrieval_before_after_update

,query,before_retrieved_sources,after_retrieved_sources,changed
0,"Что, согласно легенде, упало на голову Ньютону, натолкнув его на открытие закона всемирного тяготения?","[science_04, science_04, science_04, science_10, science_10, science_08, science_08]","[science_04, science_04, science_04, science_10, new_sci_01, science_10, science_08]",True
1,Какой именно пример (животное и его признак) приводит Ламарк в своей эволюционной теории как доказательство упражнения органов?,"[science_02, science_13, science_04, science_05, science_04, science_05, science_15]","[new_sci_02, new_sci_02, science_02, science_13, science_04, new_sci_02, science_05]",True
2,"Какую элементарную частицу открыл Джозеф Джон Томсон, и во сколько раз она легче атома водорода?","[science_15, science_14, science_05, science_05, science_06, science_05, science_07]","[new_sci_03, science_15, new_sci_03, science_14, science_05, science_05, science_06]",True
3,Какую именно руку (правую или левую) использует правило Флеминга для определения направления индукционного тока?,"[science_04, science_04, science_12, science_04, science_12, science_04, science_12]","[new_sci_04, new_sci_04, new_sci_04, science_04, science_04, science_12, science_04]",True


In [56]:
class MiniRAG:

    def __init__(self, embedder, search_index, chunks_df):
        self.embedder = embedder
        self.search_index = search_index
        self.chunks_df = chunks_df

    def answer(self, query: str, top_k: int = 3) -> dict:
        """
        Основной метод RAG:
        1. Получить запрос пользователя
        2. Извлечь top-k релевантных фрагментов
        3. Собрать контекст
        4. Сформировать ответ
        5. Вернуть ответ + источники
        """
        query_vec = self.embedder.encode_queries([query])
        scores, indices = self.search_index.search(query_vec, top_k=top_k)

        context_parts = []
        sources = []
        retrieved_chunks = []

        for idx, score in zip(indices[0], scores[0]):
            chunk = self.chunks_df.iloc[int(idx)]
            context_parts.append(chunk["chunk_text"])
            sources.append(chunk["doc_id"])
            retrieved_chunks.append(chunk["chunk_text"])

        if scores[0][0] > 0.3:
            answer = self._format_answer(query, retrieved_chunks, sources)
        else:
            answer = "Не найдено релевантной информации для ответа на этот вопрос."

        return {
            "query": query,
            "answer": answer,
            "sources": sources,
            "retrieved_chunks": retrieved_chunks,
            "top_scores": [round(float(s), 4) for s in scores[0]]
        }

    def _format_answer(self, query: str, chunks: List[str], sources: List[dict]) -> str:
        answer_parts = []
        if chunks:
            answer_parts.append(f"{chunks[0][:300]}")
        return "\n".join(answer_parts)

rag = MiniRAG(embedder, search_index, chunks_df)

In [57]:
test_queries = [
    "В каком году опубликована специальная теория относительности Эйнштейна?",
    "Как назывался корабль Дарвина?",
    "Кто сделал рентгеновские снимки ДНК для Уотсона и Крика?",
    "Какие три элемента предсказал Менделеев?",
    "Какие элементы предсказал Менделеев?",
]

answers = []

for query in test_queries:

    result = rag.answer(query, top_k=3)

    answers.append({
        "question": query,
        "answer": result["answer"],
        "retrieved_sources": result["sources"]
    })

rag_examples = pd.DataFrame(answers)
rag_examples.to_csv("./artifacts/rag_examples.csv")
rag_examples

,question,answer,retrieved_sources
0,В каком году опубликована специальная теория относительности Эйнштейна?,"Фундаментальная физическая теория пространства-времени и гравитации, пришедшая на смену ньютоновской механике. Специальная теория относительности, опубликованная в 1905 году, постулирует, что скорость света","[science_01, science_04, science_01]"
1,Как назывался корабль Дарвина?,"вид постепенно меняется. Дарвин совершил кругосветное путешествие на корабле «Бигль», где особенно его поразили вьюрки на Галапагосских островах: у разных видов клювы","[science_02, science_08, science_06]"
2,Кто сделал рентгеновские снимки ДНК для Уотсона и Крика?,"снимки Розалинд Франклин, на которых была видна характерная спиральная форма. Открытие объяснило, как происходит репликация ДНК перед делением клетки и как возникают","[science_03, science_03, science_06]"
3,Какие три элемента предсказал Менделеев?,"валентность. Например, он предсказал существование галлия (назвал его экаалюминий), скандия (экабор) и германия (экасилиций). Когда эти элементы были открыты в течение следующих","[science_05, science_05, science_05]"
4,Какие элементы предсказал Менделеев?,"— атомного номера). Менделееву приснился знаменитый сон, в котором он увидел таблицу, где элементы выстроились в ряды и столбцы по возрастанию веса","[science_05, science_05, science_05]"
